# 🎬 London Beer Flood 1814 - GPU Cloud Render & Benchmark

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/arjunmalpani/london-beer-flood-1814/blob/main/render_colab.ipynb)

### ⚡ Applied Performance Audit Optimizations:
- **Shadow Map**: Reduced from 4096×4096 to 1024×1024 (16× fewer shadow depth pixels).
- **Postprocessing**: Heavy 112-tap LensBlur, second-pass InkPass, and Bloom bypassed by default (fully customizable via `--props`).
- **Audio**: Removed redundant 20MB WAV decode inside Chromium; pure 320k stereo AAC muxed via FFmpeg.
- **Concurrency**: Tuned to 1–2 workers to prevent CPU/GPU thread contention on the 2-vCPU Colab instance.
- **Hardware Acceleration**: Enabled ANGLE EGL + NVENC.

In [ ]:
# Cell 1: Check NVIDIA GPU
!nvidia-smi

In [ ]:
# Cell 2: Pull latest code & install all Linux GPU dependencies
import os
if os.path.exists('london-beer-flood-1814'):
    %cd london-beer-flood-1814
    !git pull origin main
else:
    !git clone https://github.com/arjunmalpani/london-beer-flood-1814.git
    %cd london-beer-flood-1814

# Node 20 LTS & system packages for GPU headless Chrome
!curl -fsSL https://deb.nodesource.com/setup_20.x | sudo -E bash -
!sudo apt-get update && sudo apt-get install -y nodejs ffmpeg xvfb \
    libvulkan1 libgbm-dev libxcomposite1 libxdamage1 libasound2 \
    libnss3 libatk1.0-0 libatk-bridge2.0-0 libxrandr2 libcups2

!npm install --legacy-peer-deps

In [ ]:
# Cell 3: Diagnostic - Check Chrome GPU Status
# This verifies whether WebGL is Hardware Accelerated on the T4 GPU
!xvfb-run -a -s "-screen 0 1920x1080x24" npx remotion gpu --gl=angle-egl || \
 xvfb-run -a -s "-screen 0 1920x1080x24" npx remotion gpu --gl=angle

In [ ]:
# Cell 4: 10-Second Benchmark (300 frames) as requested by Performance Audit
# Measures exact seconds-per-frame with 1024 shadows + optimized shaders
import time
t0 = time.time()

!mkdir -p out
!xvfb-run -a -s "-screen 0 1920x1080x24" npx remotion render src/index.ts BeerFlood out/bench_300.mp4 \
    --frames=0-300 \
    --gl=angle-egl \
    --scale=0.6667 \
    --concurrency=1 \
    --hardware-acceleration=if-possible \
    --overwrite

dt = time.time() - t0
print(f"\n⏱️ Benchmark: 300 frames rendered in {dt:.1f}s ({dt/300:.3f} s/frame)!")

In [ ]:
# Cell 5: Full Render (All 2,092 frames)
# Uses optimized 1024 shadows, concurrency=2, 720p hardware render
import time
t0 = time.time()

!xvfb-run -a -s "-screen 0 1920x1080x24" npx remotion render src/index.ts BeerFlood out/raw_video.mp4 \
    --gl=angle-egl \
    --scale=0.6667 \
    --concurrency=2 \
    --hardware-acceleration=if-possible \
    --timeout=120000 \
    --overwrite

dt = time.time() - t0
print(f"\n🎉 Full render of 2,092 frames finished in {dt/60:.2f} minutes!")

In [ ]:
# Cell 6: Upscale to 1080x1920 (Lanczos) & Mux Master Audio Mix (-15 LUFS)
!ffmpeg -y -i out/raw_video.mp4 -i public/audio/mix.wav \
    -vf "scale=1080:1920:flags=lanczos" \
    -map 0:v:0 -map 1:a:0 \
    -c:v libx264 -crf 18 -preset fast \
    -c:a aac -b:a 320k \
    -shortest \
    out/final_beer_flood_1814.mp4

!ls -lh out/final_beer_flood_1814.mp4
print("\n Video ready with pristine 1080x1920 visual quality and -15 LUFS master audio!")

In [ ]:
# Cell 7: Download the final video to your computer
from google.colab import files
files.download('out/final_beer_flood_1814.mp4')